# Module 4 Practical — The Tuning Lab 🎛️

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 1 · Module 4: Tokens, temperature, context windows & model parameters**

---

### What you'll do in the next 30 minutes

| # | Experiment | Knob |
|---|-----------|------|
| 1 | Count tokens offline | `tiktoken` — free, no API calls |
| 2 | Temperature sweep | `temperature` 0 → 0.8 → 1.5, three runs each |
| 3 | Get cut off on purpose | `max_tokens` + `finish_reason` |
| 4 | Fight repetition | `frequency_penalty` |
| 5 | Hit the memory wall | context windows + a sliding-window trimmer |
| 🏁 | **SmartBot v4** | Two-speed bot: cold classifier + warm chat |

> 🎯 **Today's goal in one line:** never again ship an API call with settings you didn't choose.

## Step 0 — Setup 🔑

Same OpenAI key workflow as yesterday. New: `tiktoken`, OpenAI's tokenizer library — it runs **offline**, so Experiment 1 costs nothing.

In [ ]:
%pip install -q -U openai tiktoken

In [ ]:
from getpass import getpass
from openai import OpenAI

API_KEY = getpass("Paste the OpenAI API key and press Enter: ")
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o-mini"   # swap freely — check client.models.list() (Day 3)

print("✅ Ready. Model:", MODEL)

---
## Experiment 1 — Count tokens offline with tiktoken 🔢

On Day 1 we asked the API to count tokens. Professionals count **before** sending — to predict cost and check fit against the context window — with a local tokenizer:

In [ ]:
import tiktoken

# get the tokenizer used by our model family
enc = tiktoken.get_encoding("o200k_base")

sentence = "SmartBot replied instantly!"
token_ids = enc.encode(sentence)
print("Token IDs:", token_ids)
print("Pieces  :", [enc.decode([t]) for t in token_ids])
print("Count   :", len(token_ids), "tokens")

In [ ]:
# Rules of thumb, verified live:
tests = [
    "Hi!",
    "The quick brown fox jumps over the lazy dog.",
    "Supercalifragilisticexpialidocious",
    "मुझे रोबोटिक्स बहुत पसंद है",
    "def add(a, b): return a + b",
]
for t in tests:
    n = len(enc.encode(t))
    print(f"{n:>3} tokens | {len(t.split()):>2} words | {t}")

In [ ]:
# The practical use: how many tokens is SmartBot's system prompt?
SMARTBOT_V2 = """# ROLE
You are SmartBot, the customer assistant of SmartMart retail store, Pimple Saudagar, Pune.

# CONTEXT (the only facts you know — never invent others)
- Store hours: 9 AM - 9 PM, all 7 days
- Returns: within 7 days WITH receipt; without receipt -> store credit only
- Delivery: free above Rs. 999, else Rs. 49; standard 2-4 days in Pune
- Current offer: 10% off kitchen appliances till Sunday
- Contact for escalation: pmssupport@smartmart.example / (+91) 9960664674

# TASK
Answer customer questions about the store, orders, deliveries and policies.

# FORMAT
- Maximum 3 sentences per reply
- If the customer sounds angry, first sentence must acknowledge their frustration

# CONSTRAINTS
- If you don't know something (e.g. live stock, order tracking), say exactly:
  "I'll need to check our system for that — may I have your order number?"
- Politely refuse anything unrelated to SmartMart.
- Never invent prices, stock levels or delivery dates.
"""

n = len(enc.encode(SMARTBOT_V2))
print(f"System prompt: {n} tokens — re-sent with EVERY message (Day 3 lesson!)")
print(f"At 5,000 calls/day that's {n*5000:,} tokens/day before any customer speaks.")

**✏️ Your turn:** paste your Day-2 homework's shrunken system prompt and compare token counts. Every token you saved is now measurable money.

---
## Experiment 2 — The temperature sweep 🌡️

Same prompt, three temperatures, three runs each. Watch determinism dissolve into chaos:

In [ ]:
prompt = "Write one catchy slogan for SmartMart's Prima 1.5L electric kettle."

for temp in [0.0, 0.8, 1.5]:
    print(f"===== temperature = {temp} =====")
    for run in range(3):
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=40,
        )
        print(f"  run {run+1}: {r.choices[0].message.content.strip()}")
    print()

**Read your results:**
- `0.0` → three (nearly) identical slogans. The dice is sharpened to always pick the favourite.
- `0.8` → same spirit, different words each run. The sweet spot for creative-but-sane.
- `1.5` → surprises, maybe brilliance, maybe word salad. The dice is flat; underdog tokens win.

**The mapping to remember:** classifier/extraction/policy → `0`; chat/copy → `0.6–0.9`; brainstorm → `1.2+`.

**✏️ Your turn:** run the classifier prompt from Day 2 (`"Classify: 'where is my order?'"`) at temperature 1.5 a few times. Would you ship that?

---
## Experiment 3 — Get cut off on purpose ✂️

`max_tokens` is a hard ceiling, not a "please be brief". The model gets **truncated mid-sentence** — and `finish_reason` is how your code finds out:

In [ ]:
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain why the sky is blue in full detail."}],
    max_tokens=30,   # deliberately tiny
)

choice = r.choices[0]
print("Reply         :", choice.message.content)
print()
print("finish_reason :", choice.finish_reason)   # 'length' = truncated!
print("Tokens used   :", r.usage.completion_tokens)

In [ ]:
# Why this matters for real apps: a truncated JSON is a BROKEN JSON.
import json

r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user",
               "content": 'Return JSON with keys name, city, product for: '
                          '"I am Priya from Pune, my mixer grinder is late."'}],
    max_tokens=15,   # not enough room for the full JSON
)
choice = r.choices[0]
print("Output:", choice.message.content)
print("finish_reason:", choice.finish_reason)

try:
    json.loads(choice.message.content)
    print("✅ parsed")
except json.JSONDecodeError as e:
    print("💥 json.loads crashed →", e)
    print("THIS is why production code always checks finish_reason before parsing.")

**The pattern for production:** prompt for brevity (Day 2's Format part) AND set `max_tokens` as a safety net AND check `finish_reason` before using the output. Belt, braces, and a belt for the braces.

**✏️ Your turn:** find the smallest `max_tokens` that lets the JSON call finish with `finish_reason == "stop"`.

---
## Experiment 4 — Fight repetition with penalties 🔁

The penalties are niche knobs — but when you need them, nothing else helps. Let's force repetition, then cure it:

In [ ]:
prompt = "Write 8 short bullet points praising a kettle. Start every point with a different word."

for fp in [0.0, 1.5]:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,          # low temp makes repetition MORE likely — on purpose
        frequency_penalty=fp,
        max_tokens=220,
    )
    print(f"===== frequency_penalty = {fp} =====")
    print(r.choices[0].message.content.strip())
    print()

**Compare the two lists:** at `0.0` (especially with low temperature) you often see the same sentence-openers and adjectives echoing. At `1.5`, tokens that already appeared get pushed down, so wording diversifies.

**Rule:** keep penalties at `0` unless you *observe* repetition or looping. They're medicine, not vitamins.

---
## Experiment 5 — Hit the memory wall 🧱

Day 3's question: *"the history grows every turn — what happens after 200 turns?"* Answer: you hit the **context window** (and pay more every turn on the way there). Let's measure, then fix it:

In [ ]:
def count_message_tokens(messages):
    """Approximate token count of a messages list (+4/message overhead is typical)."""
    return sum(len(enc.encode(m["content"])) + 4 for m in messages)

# simulate a long conversation
history = [{"role": "system", "content": SMARTBOT_V2}]
for i in range(60):
    history.append({"role": "user", "content": f"Customer question number {i}: is the kettle good for a family of four people?"})
    history.append({"role": "assistant", "content": f"Answer number {i}: Yes! The Prima 1.5L comfortably serves 4-6 cups per boil, perfect for a family of four."})

print(f"Messages: {len(history)} | ~{count_message_tokens(history):,} tokens")
print("Every new call now re-sends ALL of this. Cost grows linearly; eventually the window overflows.")

In [ ]:
def trim_history(messages, max_tokens=1500):
    """Sliding window: keep the system prompt + the newest exchanges within budget.
    GOLDEN RULE: never trim the system prompt — that's the bot's brain."""
    system = messages[0]                  # always keep
    rest = messages[1:]
    # walk backwards from the newest message, keeping what fits
    kept, budget = [], max_tokens - (len(enc.encode(system["content"])) + 4)
    for m in reversed(rest):
        cost = len(enc.encode(m["content"])) + 4
        if budget - cost < 0:
            break
        kept.append(m)
        budget -= cost
    return [system] + list(reversed(kept))

trimmed = trim_history(history, max_tokens=1500)
print(f"Before: {len(history)} messages, ~{count_message_tokens(history):,} tokens")
print(f"After : {len(trimmed)} messages, ~{count_message_tokens(trimmed):,} tokens")
print(f"\nFirst kept message after system: {trimmed[1]['content'][:60]}...")
print("→ oldest exchanges dropped, newest kept, system prompt untouched. ✅")

**This is the sliding-window strategy from the slides.** Its weakness: the bot genuinely forgets old turns. The smarter fix — storing knowledge *outside* the chat and retrieving only what's relevant — is called **RAG, and it's the whole of Week 2.**

---
## 🏁 Finale — SmartBot v4: the two-speed bot

Everything tuned, on purpose. v4 routes each message through **two calls with different settings**:
1. **Classify** the intent — `temperature=0`, `max_tokens=8` (ice-cold, one word, deterministic)
2. **Answer** — `temperature=0.7`, `max_tokens=200` (warm, human, capped)

Plus the trimmer from Experiment 5 and `finish_reason` checks from Experiment 3.

In [ ]:
CLASSIFY_PROMPT = """Classify the customer message into exactly one label:
ORDER_STATUS, COMPLAINT, PRODUCT_QUERY, REFUND_REQUEST, STORE_INFO, OTHER
Reply with the label only.

Message: "{msg}"
Label:"""

class SmartBotV4:
    def __init__(self, model=MODEL):
        self.model = model
        self.history = [{"role": "system", "content": SMARTBOT_V2}]

    def classify(self, text):
        r = client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": CLASSIFY_PROMPT.format(msg=text)}],
            temperature=0,      # deterministic — same label every time
            max_tokens=8,
        )
        return r.choices[0].message.content.strip()

    def say(self, text):
        intent = self.classify(text)
        self.history.append({"role": "user", "content": text})
        self.history = trim_history(self.history, max_tokens=1500)
        r = client.chat.completions.create(
            model=self.model,
            messages=self.history,
            temperature=0.7,    # warm but on-message
            max_tokens=200,
        )
        choice = r.choices[0]
        answer = choice.message.content
        if choice.finish_reason == "length":
            answer += " …(reply truncated — raise max_tokens?)"
        self.history.append({"role": "assistant", "content": answer})
        return intent, answer

bot = SmartBotV4()
for msg in ["where is my order #4521?",
            "the kettle arrived broken, this is ridiculous",
            "do you open on sunday morning?",
            "tell me a joke about cricket"]:
    intent, answer = bot.say(msg)
    print(f"🧑 {msg}")
    print(f"   [intent: {intent}]")
    print(f"🛒 {answer}")
    print("-" * 70)

In [ ]:
# 💬 Chat with v4 — type 'quit' to stop. Watch the intent tags as you chat.
while True:
    user_msg = input("You: ")
    if user_msg.strip().lower() in ("quit", "exit", "q"):
        print("👋 Week 1 toolkit complete. Tomorrow: assembly day!")
        break
    intent, answer = bot.say(user_msg)
    print(f"   [intent: {intent}]")
    print("🛒 SmartBot:", answer)

---
## 🏆 Homework (15 min — final prep for tomorrow's weekly project!)

1. **Temperature report** — run your favourite Day-2 prompt at `0 / 0.8 / 1.5`, three runs each. One paragraph: which temperature would you ship for that task, and why?
2. **Right-size the budget** — for each of v4's two calls, find the smallest `max_tokens` that never triggers `finish_reason == "length"` across 5 test messages. Note the values — they go straight into tomorrow's project.
3. **Stress the trimmer** — chat with v4 for 15+ turns, then ask it about your very first message. What happened, and why? Write two sentences connecting this to what Week 2 will fix.

### What's next — assembly day! 🏗️
**Day 5 — Weekly Project:** you'll assemble everything from this week — LLM concepts, the 5-part prompt, the OpenAI plumbing, and today's tuning — into the complete, graded **Retail Q&A Chatbot**. Bring your best system prompt and your tuned settings.

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*